### Step 1: Clone Github Repository

In [35]:
!git clone https://github.com/WesAub/Capstone.git

Cloning into 'Capstone'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 67 (delta 4), reused 53 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (67/67), 1.79 MiB | 6.60 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [36]:
%cd Capstone
!git checkout feature/preprocessing

/content/Capstone/Capstone/Capstone/Capstone/Capstone
Branch 'feature/preprocessing' set up to track remote branch 'feature/preprocessing' from 'origin'.
Switched to a new branch 'feature/preprocessing'


In [37]:
!ls

data  README.md  scripts


### Step 2: LLM Dataset Preprocessing

In [38]:
!python scripts/preprocess_to_llm_dataset.py --data_dir data

Wrote train=6880, val=860, test=860 to folder: out


### Step 3: Environment Setup & Authentication

In [39]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [40]:
# Install training stack
!pip -q install -U "trl>=0.9.6" "transformers>=4.41.0" "peft>=0.11.1" "accelerate>=0.33.0" bitsandbytes

In [41]:
import transformers
print("transformers version:", transformers.__version__)

common = dict(
    output_dir="ft_out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=25,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    report_to="none",
    eval_steps=200,
)

try:
    args = TrainingArguments(**common, evaluation_strategy="steps")
except TypeError:
    try:
        args = TrainingArguments(**common, eval_strategy="steps")
    except TypeError:
        print("No evaluation_strategy/eval_strategy supported in this transformers version. Disabling eval.")
        args = TrainingArguments(**{k:v for k,v in common.items() if k not in ["eval_steps"]})

transformers version: 5.2.0


In [42]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TRANSFORMERS_NO_CACHING_ALLOCATOR_WARMUP"] = "1"

import trl
print(trl.__version__)

0.29.0


In [43]:
!pip -q install -U huggingface_hub
from huggingface_hub import login
login()

### Step 4: Fine-tune LLM

In [44]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer
import os, torch
torch.backends.cuda.matmul.allow_tf32 = True

MODEL = "meta-llama/Llama-3.2-3B-Instruct"
os.makedirs("offload", exist_ok=True)

data_files = {"train": "out/train.jsonl", "validation": "out/val.jsonl"}
ds = load_dataset("json", data_files=data_files)

def build_text(example):
    if "text" in example and example["text"] is not None:
        return {"text": str(example["text"])}

    # prompt/completion style
    if "prompt" in example and "completion" in example:
        prompt = "" if example["prompt"] is None else str(example["prompt"])
        completion = "" if example["completion"] is None else str(example["completion"])
        return {"text": prompt + completion}

    # instruction/input/output style (alpaca-ish)
    if "instruction" in example and "output" in example:
        instr = "" if example["instruction"] is None else str(example["instruction"])
        inp = "" if example.get("input") is None else str(example.get("input"))
        out = "" if example["output"] is None else str(example["output"])

        # If there's no input, omit that section
        if inp.strip():
            text = f"### Instruction:\n{instr}\n\n### Input:\n{inp}\n\n### Response:\n{out}"
        else:
            text = f"### Instruction:\n{instr}\n\n### Response:\n{out}"
        return {"text": text}

    # question/answer style
    if "question" in example and "answer" in example:
        q = "" if example["question"] is None else str(example["question"])
        a = "" if example["answer"] is None else str(example["answer"])
        return {"text": f"### Question:\n{q}\n\n### Answer:\n{a}"}

    # src/tgt style (translation-ish)
    if "src" in example and "tgt" in example:
        src = "" if example["src"] is None else str(example["src"])
        tgt = "" if example["tgt"] is None else str(example["tgt"])
        return {"text": f"{src}\n{tgt}"}

    # fallback: stringify the whole row so training doesn't crash
    return {"text": str(example)}

ds = ds.map(build_text, remove_columns=ds["train"].column_names)

# Quick sanity check
print("Train columns:", ds["train"].column_names)
print("Example:\n", ds["train"][0]["text"][:500])

# ----------------------------
# 2) Quantization (4-bit)
# ----------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# ----------------------------
# 3) Tokenizer / Model
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    max_memory={0: "10GiB", "cpu": "48GiB"},
    offload_folder="offload",
    low_cpu_mem_usage=True,
)

# ----------------------------
# 4) LoRA config
# ----------------------------
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

# ----------------------------
# 5) Training args
# ----------------------------
args = TrainingArguments(
    output_dir="ft_out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=5e-5,
    num_train_epochs=1,
    logging_steps=25,
    save_steps=200,
    save_total_limit=2,
    fp16=False,
    report_to="none",
)

MAX_LEN = 256
tokenizer.model_max_length = MAX_LEN
tokenizer.truncation_side = "right"
tokenizer.padding_side = "right"

def formatting_func(example):
    return example["text"]

# ----------------------------
# 6) Trainer
# ----------------------------
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    peft_config=lora,
    formatting_func=formatting_func,
)
print("Model dtype:", next(model.parameters()).dtype)
trainer.train()

trainer.save_model("ft_adapter")
tokenizer.save_pretrained("ft_adapter")
print("Saved LoRA adapter to ft_adapter/")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/6880 [00:00<?, ? examples/s]

Map:   0%|          | 0/860 [00:00<?, ? examples/s]

Train columns: ['text']
Example:
 ### Instruction:
Two-Stage Amplifier Design in TSMC 65nm CMOS Technology.

Design a two-stage amplifier (5 MOSFET first stage, 2 MOSFET second stage) with Miller compensation.

Target specifications:
- Power: 3.5e-05
- Gain: 53.3
- BW_3dB: 10800
- UGB: 3.38e+06
- PM: 81.8
- GM: 28.3

Return ONLY the SPICE netlist.

### Response:
* Two-Stage Amplifier Design in TSMC 65nm CMOS Technology

* Biasing Voltages (mV)
* VINP=VINN=VIN=441 mV, VB1=877 mV

* MOSFETs (L in nm, W in um)
M1 pch net3 VINP net2


RuntimeError: Cannot access accelerator device when none is available.